# Flow types

summer4 has three kinds of flow, and this page uses each of them once. A **transition flow**
moves people from one compartment to another. An **entry flow** adds people to the model
(births or arrivals). An **exit flow** removes people from the model (deaths or departures).
Transitions never change the total population. Entries increase it and exits decrease it.

The model is an SIR model with births and deaths. We will set the rates as `Param`s, run the
model, and plot both the compartments and the total population.

```{admonition} Attribution
:class: note

Parts of this page's prose are adapted from the
[summer2 documentation](https://summer2.readthedocs.io), BSD-2-Clause,
Copyright (c) 2022, monash-emu
([monash-emu/summer2](https://github.com/monash-emu/summer2)). The code is
written for summer4.
```

## Imports and compartments

The next cell loads what we need and declares the same three compartments as the previous
page.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import (
    Compartments,
    EntryFlow,
    Everything,
    ExitFlow,
    FlowModel,
    Param,
    Property,
    PropertyMap,
    SavePlan,
    SaveRequest,
    TransitionFlow,
)
from summer4.epi import ForceOfInfection

state = Property("state", ("S", "I", "R"))
pmap = PropertyMap.from_property(state)

## Transition flows

Infection and recovery are transitions, exactly as on the previous page.

In [ ]:
model = FlowModel(pmap)
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("infection", infectious=state["I"], contact_rate=Param("contact_rate")),
    )
)
recovery_flow = model.add_flow(
    TransitionFlow("recovery", state["I"], state["R"], Param("recovery_rate"))
)

## An entry flow for births

`EntryFlow(name, destination, rate)` adds people to the destination compartment. Unlike a
transition, its rate is a **number of people per day**, not a rate per person, because there
is no source compartment to take a fraction of. Here 2 babies are born each day, and all of
them are born susceptible.

In [ ]:
births_flow = model.add_flow(EntryFlow("births", state["S"], Param("births_per_day")))

## An exit flow for deaths

`ExitFlow(name, source, rate)` removes people from the source compartment, at a rate per
person. `Everything()` means every compartment, so one flow applies natural deaths to the whole
population. A rate of 0.002 per day corresponds to an average life of 500 days, which is
short for a person but keeps the plot quick to read. A second exit flow removes extra
infectious people because of the disease.

In [ ]:
natural_death = model.add_flow(ExitFlow("natural_death", Everything(), Param("death_rate")))
disease_death = model.add_flow(ExitFlow("disease_death", state["I"], Param("disease_death_rate")))
start = model.set_initial_population({state["S"]: 990.0, state["I"]: 10.0})

## Run and plot the compartments

We compile, run for 400 days and plot the compartments. The first epidemic wave should burn
out within about 80 days, as it did on the previous page. After that the susceptible line
should climb slowly as babies are born, and the recovered line should fall slowly as those
people die. The infectious line stays near zero.

In [ ]:
params = {
    "contact_rate": 0.5,
    "recovery_rate": 0.2,
    "births_per_day": 2.0,
    "death_rate": 0.002,
    "disease_death_rate": 0.01,
}
compiled = model.compile()
plan = SavePlan(
    requests={"compartments": SaveRequest(Compartments())},
    ts=np.linspace(0.0, 400.0, 401),
)
result = compiled.run(params, t0=0.0, t1=400.0, dt=0.1, save=plan)
frame = result["compartments"].to_pandas()
frame.plot(
    title="SIR model with births and deaths",
    labels={"index": "time (days)", "value": "people"},
)

## Plot the total population

Add up the compartments at each time to get the total population. Births add 2 people a day
and natural deaths remove 0.2% of the population a day, which is also 2 people a day when
the population is 1000. So the total should dip during the epidemic, because of the disease
deaths, and then climb back slowly towards 1000.

In [ ]:
total = frame.sum(axis=1)
total.to_frame("total population").plot(
    title="Total population",
    labels={"index": "time (days)", "value": "people"},
)

## Compare with a model that has no entries or exits

To see that transitions alone keep the population fixed, we build the same model without the
entry and exit flows and plot both totals. The line for the model with only transitions
should be flat at 1000.

In [ ]:
closed = FlowModel(pmap)
closed.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("infection", infectious=state["I"], contact_rate=Param("contact_rate")),
    )
)
closed.add_flow(TransitionFlow("recovery", state["I"], state["R"], Param("recovery_rate")))
closed.set_initial_population({state["S"]: 990.0, state["I"]: 10.0})
closed_result = closed.compile().run(
    {"contact_rate": 0.5, "recovery_rate": 0.2}, t0=0.0, t1=400.0, dt=0.1, save=plan
)
pd.DataFrame(
    {
        "with births and deaths": total,
        "transitions only": closed_result["compartments"].to_pandas().sum(axis=1),
    }
).plot(
    title="Total population with and without entry and exit flows",
    labels={"index": "time (days)", "value": "people"},
)

## Things to try

- Set `births_per_day` to 0 and rerun. What happens to the total population?
- Set `births_per_day` to 4. Where does the total population level off? (Births balance
  deaths when `births_per_day = death_rate * population`.)
- Make the disease death rate zero. How does the final size of the recovered compartment
  change?
- Add `EntryFlow("importation", state["I"], 0.1)` to bring in infectious travellers each day.